## 6.4 多输入多输出通道
### 6.4.1 多输入通道
当一个输入存在多个通道时，比如RGB图像，每个通道对应一个颜色。我们面对的是一个$3 \times h \times w$的输入，其中$h$和$w$是输入图像的高和宽。那么我们需要构造一个与输入数据具有相同输入通道数的卷积核。
计算的时候，我们对每个通道输入的二维张量和卷积核的二维张量进行卷积，再对通道求和得到二维张量。

In [ ]:
import torch
from torch import nn
def corr2d_multi_in(X : torch.tensor, K : torch.tensor):
    # 先遍历“X”和“K”的第0个维度（通道维度），再把它们加在一起
    Y = sum(nn.functional.conv2d(x.unsqueeze(0).unsqueeze(0), k.unsqueeze(0).unsqueeze(0)) for x, k in zip(X, K))
    return Y.reshape(Y.shape[2:])

X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]], 
                  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])

corr2d_multi_in(X, K)


### 6.4.2 多输出通道

In [ ]:
def corr2d_multi_in_out(X, K):
    # 迭代“K”的第0个维度，每次都对输入“X”执行互相关运算。
    # 最后将所有结果都叠加在一起
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

K = torch.stack((K, K + 1, K + 2), 0)
K.shape

In [ ]:
corr2d_multi_in_out(X, K)

### 6.4.3 $1 \times 1$ 卷积层
这种卷积层失去了识别相邻元素间相互作用的能力，唯一的计算发生在通道上。

In [12]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))

Y1 = corr2d_multi_in_out_1x1(X, K)
Y2= corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6